In [ ]:
---
title: "Advanced Agent"
author: "Jaume Amores"
format: html
---

# 1. MCP

- [video](https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/lessons/71234850-lesson-1-foundational-models)

## 1.1. MCP

- [notebook 1](https://github.com/langchain-ai/lca-lc-foundations/blob/main/notebooks/module-2/2.1_mcp.ipynb)

### Advantages of using MCP

Many mcp servers developed in open source repos or third party providers which can be just plugged into our graph.

### Components

- python script file defining MCP components
    - with `FastMCP` class 
    - with decorators @mcp.xxx:
        - tool (e.g., websearch)
        - prompt (e.g., system prompt)
        - resources (e.g., files that can be downloaded from github)

- In notebook:
    - Create `MultiServerMCPClient`, and pass:
        - transport => script: stdio
        - command => python, uvx (install and run python library)
        - args => either path to script, or python library to install and run, plus other args (time in Los Angeles)

    - get the components from client in notebook, then pass them to `create_agent`


## 1.2. Travel Agent

- [notebook 2](https://github.com/langchain-ai/lca-lc-foundations/blob/main/notebooks/module-2/2.1_travel_agent.ipynb)

# 2. Context and State

[video](https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/lessons/71234862-lesson-2-context-and-state)


## 2.1. Runtime context 

[notebook](https://github.com/langchain-ai/lca-lc-foundations/blob/main/notebooks/module-2/2.2_runtime_context.ipynb)

### Implementation

- Context schema class: can have defaults
- create_agent: `context_schema=MyContextClass`
- invoke: `context=MyContextClass()` or `context=MyContextClass(field=my_context_value,...)` to pass context at starting point
- tool: `runtime:ToolRuntime`, `runtime.context.my_field`

### Notes

- Context passed in the context object cannot be accessed directly by the agent. 
    - It needs to be given through tool calls, which read the context fields and give the appropriate context information based on them. 
    - For example, we can just declare tools for each field we want the agent to know and then make each tool retrieve the value of this field. Just by passing the tools at creation time the agent will know how to get these fields when asked (given appropriate description of tool and name of tool)
- Context does not change as a result of conversation with user. For this we need to access state.

## 2.2. State

[notebook](https://github.com/langchain-ai/lca-lc-foundations/blob/main/notebooks/module-2/2.2_state.ipynb)

### Update context in tool

- What we do actually is to update the *state*
Return Command object

```python
@tool
def my_tool (my_field, runtime: ToolRuntime):
return Command(
    update={
        "my_field":my_field,
        "messages": [
            ToolMessage(
                "Updated xxx",
                tool_call_id=runtime.tool_call_id
            )
        ]
    }
)
```

- At creation time we pass in a *state* schema: `state_schema=MyStateClass`
- `MyStateClass` is usually a subclass of `AgentState`
- We can update the value in `invoke` call:
    - either through message content: 
    ```python
    agent.invoke(
        {"messages": [HumanMessage(content="field value is x")]},
        ...
    )
    ```
    - or passing value as additional field of input dictionary
    ```python
    agent.invoke (
        {
            "messages": [HumanMessage(content="hello")],
            my_field: my_value
        }
    )
    ```
- In create_agent, we need to pass a checkpointer for short-term memory
- In invoke, we need to pass the config dict with thread value


### Read state from tool

through state variable of runtime object: `runtime.state["my_field"]`



# Multi-agent

- [video](https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/lessons/71234863-lesson-3-multi-agent-systems)
- [notebook](https://github.com/langchain-ai/lca-lc-foundations/blob/main/notebooks/module-2/2.3_multi_agent.ipynb)

## Issues of single agent

In complex tasks, if a single agent needs to handle

Example: high quality market research report requiring: 

- Long workflow
- Many specialist skills at different times: proposing an outline, researching on the web, validating sources, writing the report, editing.

If we use single agent:
- Too many tools to choose from.
- Context window overflows with information.

A multi-agent system can:
- Breakdown this complex task into multiple specialized agents.
- These can work *together* to solve the problem.

## Simple implementation

- Each subagent calls a tool (could be set of tools)
- Each subagent is wrapped in a tool function
- The main agent is given the list of tools, where each *tool* is actually a subagent.